# Análisis de Predicciones — Grado

Este notebook analiza las predicciones activas almacenadas en `PMAT_PREDICTION` para los alumnos de **Grado**, cargando los datos directamente desde Oracle.

**Secciones:**
1. Carga de datos desde Oracle
2. Resumen global de predicciones
3. Rendimiento del modelo *(solo si TARGET_REAL está disponible)*
4. Predicciones por titulación (Top 5)
5. Evolución de la probabilidad por etapa
6. Importancia de variables del modelo
7. Análisis SHAP (variables explicativas por predicción)

## 0. Setup y carga de datos

In [ ]:
import sys
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

# Ajustar path para importar módulos del proyecto
PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from oracle_connector import OracleConnector

# ── Constante: qué modelo analizar ───────────────────────────────────────────
TIPO = 'grado'        # 'grado' | 'master'
MODELO_FILTRO = 'grado'  # filtra MODELO LIKE '%grado%'
RUTA_MODELO = PROJECT_ROOT / 'models' / 'modelo_final_grado.pkl'

In [ ]:
conn = OracleConnector()

# ── 1. Predicciones ──────────────────────────────────────────────────────────
df_pred_raw = pd.DataFrame(conn.read_table('PMAT_PREDICTION'))
print(f'PMAT_PREDICTION: {len(df_pred_raw):,} filas | {df_pred_raw.shape[1]} columnas')

# Filtrar solo el modelo de Grado
df_pred = df_pred_raw[
    df_pred_raw['MODELO'].str.contains(MODELO_FILTRO, case=False, na=False)
].copy()
print(f'Después del filtro ({MODELO_FILTRO}): {len(df_pred):,} filas')

# ── 2. Dataset limpio (para TITULACION y features) ───────────────────────────
df_ds = pd.DataFrame(conn.read_table('DATASET_LIMPIO'))
# Filtrar grado (excluir MASTER)
df_ds = df_ds[~df_ds['TITULACION'].str.contains('MASTER', case=False, na=False)].copy()
print(f'DATASET_LIMPIO (Grado): {len(df_ds):,} filas')

# ── 3. Join para añadir TITULACION y fecha de creación ───────────────────────
df = df_pred.merge(
    df_ds[['ID', 'TITULACION', 'CREATEDDATE']].drop_duplicates('ID'),
    left_on='OPP_ID', right_on='ID',
    how='left'
)

# Normalizar columnas
df['PROBABILIDAD']  = pd.to_numeric(df['PROBABILIDAD'], errors='coerce')
df['TARGET_PRED']   = pd.to_numeric(df['TARGET_PRED'],  errors='coerce').astype('Int64')
df['TARGET_REAL']   = pd.to_numeric(df['TARGET_REAL'],  errors='coerce')
df['etapa_compuesta'] = (
    df['ETAPA'].fillna('NA').astype(str).str.strip() + '__' +
    df['SUBETAPA'].fillna('NA').astype(str).str.strip()
)
df['FECHA_PRED'] = pd.to_datetime(df['FECHA_PRED'], errors='coerce')

print(f'\nDataset final: {len(df):,} filas × {df.shape[1]} columnas')
df.head(3)

## 1. Resumen global de predicciones

In [ ]:
n_total     = len(df)
n_opp       = df['OPP_ID'].nunique()
n_matricula = (df['TARGET_PRED'] == 1).sum()
n_no_matr   = (df['TARGET_PRED'] == 0).sum()
pct_matr    = n_matricula / n_total * 100 if n_total else 0
tiene_real  = df['TARGET_REAL'].notna().sum()

print('=' * 55)
print('   RESUMEN GLOBAL — PREDICCIONES GRADO')
print('=' * 55)
print(f'  Registros totales (etapas)  : {n_total:>8,}')
print(f'  Oportunidades únicas        : {n_opp:>8,}')
print(f'  Predicción = MATRÍCULA (1)  : {n_matricula:>8,}  ({pct_matr:.1f}%)')
print(f'  Predicción = NO MATRÍCULA(0): {n_no_matr:>8,}  ({100-pct_matr:.1f}%)')
print(f'  Con TARGET_REAL disponible  : {tiene_real:>8,}')
print(f'  Probabilidad media          : {df["PROBABILIDAD"].mean():>8.3f}')
print(f'  Confianza media             : {pd.to_numeric(df["CONFIANZA"], errors="coerce").mean():>8.3f}')
print('=' * 55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histograma de probabilidades ─────────────────────────────────────────────
colores = df['TARGET_PRED'].map({1: '#2ecc71', 0: '#e74c3c'})
axes[0].hist(
    [df.loc[df['TARGET_PRED']==1, 'PROBABILIDAD'],
     df.loc[df['TARGET_PRED']==0, 'PROBABILIDAD']],
    bins=30, stacked=True, color=['#2ecc71', '#e74c3c'],
    label=['Predicción: Matrícula', 'Predicción: No Matrícula'], edgecolor='white'
)
axes[0].set_title('Distribución de Probabilidades', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Probabilidad de Matrícula')
axes[0].set_ylabel('Nº de registros')
axes[0].legend()
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.6, label='Umbral 0.5')

# ── Donut: proporción predicha ────────────────────────────────────────────────
valores = [n_matricula, n_no_matr]
etiquetas = [f'Matrícula\n{n_matricula:,} ({pct_matr:.1f}%)',
             f'No Matrícula\n{n_no_matr:,} ({100-pct_matr:.1f}%)']
wedge_props = {'width': 0.5, 'edgecolor': 'white'}
axes[1].pie(valores, labels=etiquetas, colors=['#2ecc71', '#e74c3c'],
            wedgeprops=wedge_props, startangle=90,
            textprops={'fontsize': 11})
axes[1].set_title('Proporción de Predicciones', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Rendimiento del modelo

> Esta sección requiere que **TARGET_REAL** esté disponible (resultados históricos cerrados).
> Si los registros son solo predicciones activas, las celdas mostrarán un aviso.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)

df_eval = df.dropna(subset=['TARGET_REAL']).copy()
df_eval['TARGET_REAL'] = df_eval['TARGET_REAL'].astype(int)
df_eval['TARGET_PRED'] = df_eval['TARGET_PRED'].astype(int)

if len(df_eval) < 50:
    print(f'⚠️  Solo {len(df_eval)} registros con TARGET_REAL. '
          'Se necesitan al menos 50 para métricas fiables.')
else:
    # Colapsar por oportunidad única (ID)
    df_opp = df_eval.groupby('OPP_ID').agg(
        target=('TARGET_REAL', 'max'),
        pred=('TARGET_PRED', 'max'),
        prob=('PROBABILIDAD', 'mean')
    ).reset_index()

    acc  = accuracy_score(df_opp['target'], df_opp['pred'])
    prec = precision_score(df_opp['target'], df_opp['pred'], zero_division=0)
    rec  = recall_score(df_opp['target'], df_opp['pred'], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(df_opp['target'], df_opp['pred']).ravel()
    especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0

    try:
        auc = roc_auc_score(df_opp['target'], df_opp['prob'])
        auc_str = f'{auc:.4f}'
    except Exception:
        auc_str = 'N/A'

    print('-' * 52)
    print('   MÉTRICAS DEL MODELO (Oportunidades únicas)')
    print('-' * 52)
    print(f'  Exactitud (Accuracy)      : {acc:.2%}')
    print(f'  AUC-ROC                   : {auc_str}')
    print(f'  Precisión (pred=1 acierta): {prec:.2%}')
    print(f'  Sensibilidad (Recall)     : {rec:.2%}')
    print(f'  Especificidad             : {especificidad:.2%}')
    print(f'  Evaluados                 : {len(df_opp):,} oportunidades')
    print('-' * 52)

In [ ]:
if len(df_eval) >= 50:
    cm = confusion_matrix(df_opp['target'], df_opp['pred'])

    # Probabilidad media por cuadrante
    prob_cuadrante = []
    for real in [0, 1]:
        fila = []
        for pred_val in [0, 1]:
            mask = (df_opp['target'] == real) & (df_opp['pred'] == pred_val)
            fila.append(df_opp.loc[mask, 'prob'].mean())
        prob_cuadrante.append(fila)

    labels = [
        [f"{cm[0,0]:,}\nProb: {prob_cuadrante[0][0]:.0%}",
         f"{cm[0,1]:,}\nProb: {prob_cuadrante[0][1]:.0%}"],
        [f"{cm[1,0]:,}\nProb: {prob_cuadrante[1][0]:.0%}",
         f"{cm[1,1]:,}\nProb: {prob_cuadrante[1][1]:.0%}"]
    ]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False,
                xticklabels=['Pred: NO', 'Pred: SÍ'],
                yticklabels=['Real: NO', 'Real: SÍ'],
                annot_kws={'size': 13, 'weight': 'bold'})
    plt.title('Matriz de Confusión — Oportunidades únicas\n(con probabilidad media por cuadrante)',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Matriz de confusión no disponible (TARGET_REAL insuficiente).')

## 3. Predicciones por titulación (Top 5)

In [ ]:
# Una fila por oportunidad (última etapa)
df_uniq = (
    df.sort_values('FECHA_PRED')
    .groupby('OPP_ID')
    .agg(TITULACION=('TITULACION', 'first'),
         TARGET_PRED=('TARGET_PRED', 'last'),
         TARGET_REAL=('TARGET_REAL', 'max'),
         PROBABILIDAD=('PROBABILIDAD', 'last'))
    .reset_index()
)

top5 = df_uniq.groupby('TITULACION')['OPP_ID'].nunique().nlargest(5).index
df_top5 = df_uniq[df_uniq['TITULACION'].isin(top5)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Barras: tasa de predicción positiva por titulación ───────────────────────
tasa_pred = (
    df_top5.groupby('TITULACION')['TARGET_PRED']
    .mean()
    .reindex(top5)
    .reset_index()
)
ax = sns.barplot(x='TITULACION', y='TARGET_PRED', data=tasa_pred,
                 palette='viridis', ax=axes[0])
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Tasa de Predicción Positiva por Titulación (Top 5)',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('% predicho como Matrícula')
axes[0].set_ylim(0, 1.15)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right')

# ── Volumen de oportunidades por titulación ───────────────────────────────────
vol = df_top5.groupby('TITULACION')['OPP_ID'].nunique().reindex(top5).reset_index()
vol.columns = ['TITULACION', 'N']
sns.barplot(x='TITULACION', y='N', data=vol, palette='mako', ax=axes[1])
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('Volumen de Oportunidades por Titulación (Top 5)',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nº de oportunidades')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=40, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
ax = sns.violinplot(
    x='TITULACION', y='PROBABILIDAD',
    data=df_top5, palette='viridis',
    inner='quartile', cut=0, order=top5
)

counts_tit = df_top5.groupby('TITULACION')['OPP_ID'].nunique()
for i, tit in enumerate(top5):
    n = counts_tit.get(tit, 0)
    ax.text(i, 1.05, f'n={n:,}', ha='center', va='bottom',
            fontweight='bold', fontsize=10)

plt.title('Top 5 Titulaciones: Distribución de Probabilidad de Matrícula',
          fontsize=14, fontweight='bold', pad=25)
plt.ylabel('Probabilidad de Matrícula')
plt.xlabel('Titulación')
plt.ylim(0, 1.15)
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Evolución de la probabilidad por etapa

In [ ]:
# Calcular orden de etapas automáticamente por fecha media de aparición
df['FECHA_PRED'] = pd.to_datetime(df['FECHA_PRED'], errors='coerce')

primeras = (
    df.groupby(['OPP_ID', 'etapa_compuesta'])['FECHA_PRED']
    .min().reset_index()
)
primeras['ranking'] = (
    primeras.groupby('OPP_ID')['FECHA_PRED'].rank(method='first')
)
orden = (
    primeras.groupby('etapa_compuesta')['ranking']
    .mean().sort_values().reset_index()
)
orden['etapa_ordinal_num'] = range(len(orden))
mapa_orden = dict(zip(orden['etapa_compuesta'], orden['etapa_ordinal_num']))

df['etapa_ordinal_num'] = df['etapa_compuesta'].map(mapa_orden)
etapas_ordenadas = orden['etapa_compuesta'].tolist()
df['etapa_compuesta'] = pd.Categorical(
    df['etapa_compuesta'], categories=etapas_ordenadas, ordered=True
)
print(f'Etapas detectadas: {len(etapas_ordenadas)}')

In [ ]:
plt.figure(figsize=(18, 8))
ax = sns.violinplot(
    x='etapa_compuesta', y='PROBABILIDAD',
    data=df, palette='viridis',
    inner='quartile', bw_adjust=0.5, cut=0
)

counts_etapa = df['etapa_compuesta'].value_counts().reindex(etapas_ordenadas)
for i, count in enumerate(counts_etapa):
    if pd.notna(count) and count > 0:
        ax.text(i, 1.06, f'n={int(count):,}', ha='center', va='bottom',
                fontweight='bold', size=8)

plt.title('Distribución de Probabilidad de Matrícula por Etapa',
          fontsize=14, fontweight='bold', pad=30)
plt.xlabel('Etapa__Subetapa (orden cronológico)', fontsize=11)
plt.ylabel('Probabilidad de Matrícula', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.ylim(0, 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Métricas por etapa (mínimo 20 casos)
df_stats = df.groupby('etapa_compuesta', observed=True).agg(
    tasa_pred=('TARGET_PRED', lambda x: (x == 1).mean()),
    prob_media=('PROBABILIDAD', 'mean'),
    etapa_ordinal_num=('etapa_ordinal_num', 'mean'),
    num_casos=('OPP_ID', 'count')
).reset_index()
df_stats = df_stats[df_stats['num_casos'] >= 20].sort_values('etapa_ordinal_num')

if df_stats.empty:
    print('No hay etapas con ≥20 casos')
else:
    fig, ax = plt.subplots(figsize=(16, 7))
    x = np.arange(len(df_stats))
    ax.bar(x, df_stats['tasa_pred'], color='royalblue', alpha=0.7,
           label='Tasa predicha como Matrícula')
    ax2 = ax.twinx()
    ax2.plot(x, df_stats['prob_media'], marker='o', color='firebrick',
             linewidth=2.5, markersize=8, label='Probabilidad media')

    for i, row in df_stats.reset_index(drop=True).iterrows():
        ax.text(i, row['tasa_pred'] + 0.02, f"n={int(row['num_casos']):,}",
                ha='center', fontsize=8, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(df_stats['etapa_compuesta'], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Tasa predicha como Matrícula', fontsize=11)
    ax2.set_ylabel('Probabilidad media', fontsize=11, color='firebrick')
    ax.set_ylim(0, 1.2)
    ax2.set_ylim(0, 1.2)
    ax.set_title('Predicciones y Probabilidad Media por Etapa del Funnel',
                 fontsize=14, fontweight='bold', pad=15)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Importancia de variables del modelo

> Requiere el fichero `models/modelo_final_grado.pkl`.

In [ ]:
if not RUTA_MODELO.exists():
    print(f'⚠️  Modelo no encontrado en: {RUTA_MODELO}')
    print('   Sube el fichero .pkl a la carpeta models/ y vuelve a ejecutar.')
else:
    from pycaret.classification import load_model
    # PyCaret guarda el modelo sin extensión .pkl
    modelo = load_model(str(RUTA_MODELO).replace('.pkl', ''))

    last_step = list(modelo.named_steps.keys())[-1]
    estimador = modelo.named_steps[last_step]

    importancia_vals = estimador.feature_importances_
    n_feat = len(importancia_vals)

    try:
        feature_names = list(modelo.feature_names_in_)
        if len(feature_names) != n_feat:
            feature_names = feature_names[:n_feat] if len(feature_names) > n_feat \
                else feature_names + [f'Feature_{i}' for i in range(n_feat - len(feature_names))]
    except AttributeError:
        feature_names = [f'Variable_{i}' for i in range(n_feat)]

    importancias = pd.Series(importancia_vals, index=feature_names)
    top_n = min(15, n_feat)
    top_imp = importancias.nlargest(top_n).sort_values()

    plt.figure(figsize=(10, 7))
    colores = plt.cm.viridis(np.linspace(0.2, 0.9, top_n))
    top_imp.plot(kind='barh', color=colores)
    plt.title(f'Top {top_n} Variables más Importantes — Modelo Grado',
              fontsize=14, fontweight='bold')
    plt.xlabel('Importancia Relativa')
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    print('\nTop 10 variables más importantes:')
    for var, imp in importancias.nlargest(10).items():
        print(f'  {var:<45} {imp:.4f}')

## 6. Análisis SHAP — Variables explicativas por predicción

La columna `EXPLICACION` contiene el top-3 de variables SHAP para cada predicción en formato JSON.
Aquí se agregan para ver qué variables explican más frecuentemente las predicciones positivas y negativas.

In [ ]:
def parsear_shap(explicacion_str):
    """Parsea el JSON de EXPLICACION y devuelve lista de dicts {feature, value, shap_value}."""
    if pd.isna(explicacion_str) or not explicacion_str:
        return []
    try:
        data = json.loads(str(explicacion_str))
        return data if isinstance(data, list) else []
    except Exception:
        return []

# Expandir todas las explicaciones
filas_shap = []
for _, row in df.iterrows():
    items = parsear_shap(row.get('EXPLICACION'))
    for item in items:
        filas_shap.append({
            'OPP_ID':      row['OPP_ID'],
            'TARGET_PRED': row['TARGET_PRED'],
            'PROBABILIDAD': row['PROBABILIDAD'],
            'feature':     item.get('feature', ''),
            'value':       item.get('value', None),
            'shap_value':  item.get('shap_value', 0),
        })

df_shap = pd.DataFrame(filas_shap)
print(f'Registros SHAP parseados: {len(df_shap):,}')
df_shap.head(5)

In [ ]:
if df_shap.empty:
    print('⚠️  No hay datos SHAP disponibles en EXPLICACION.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, pred_val, titulo, color in [
        (axes[0], 1, 'Predicción: MATRÍCULA', '#2ecc71'),
        (axes[1], 0, 'Predicción: NO MATRÍCULA', '#e74c3c'),
    ]:
        subset = df_shap[df_shap['TARGET_PRED'] == pred_val]
        if subset.empty:
            ax.set_title(f'{titulo}\n(sin datos)', fontsize=12)
            continue

        # Frecuencia + SHAP medio por variable
        resumen = (
            subset.groupby('feature')
            .agg(frecuencia=('feature', 'count'),
                 shap_medio=('shap_value', lambda x: x.abs().mean()))
            .sort_values('frecuencia', ascending=False)
            .head(10)
            .sort_values('frecuencia')
        )

        bars = ax.barh(resumen.index, resumen['frecuencia'], color=color, alpha=0.8)
        ax.set_title(f'Variables más frecuentes en SHAP\n{titulo}',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Frecuencia (aparece en Nº predicciones)')

        for bar, (_, row) in zip(bars, resumen.iterrows()):
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f"SHAP: {row['shap_medio']:.3f}",
                    va='center', fontsize=9)

        ax.grid(axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

In [ ]:
if not df_shap.empty:
    # SHAP medio por variable (valor absoluto) — visión global
    shap_global = (
        df_shap.groupby('feature')['shap_value']
        .apply(lambda x: x.abs().mean())
        .sort_values(ascending=True)
        .tail(12)
    )

    plt.figure(figsize=(10, 6))
    colores = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(shap_global)))
    shap_global.plot(kind='barh', color=colores)
    plt.title('Impacto SHAP medio por variable — Grado\n(promedio del valor absoluto)',
              fontsize=13, fontweight='bold')
    plt.xlabel('|SHAP| medio')
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
if not df_shap.empty:
    print('Ejemplos de predicciones con mayor probabilidad (Matrícula):')
    top_pred = df[df['TARGET_PRED'] == 1].nlargest(5, 'PROBABILIDAD')[['OPP_ID', 'ETAPA', 'SUBETAPA', 'PROBABILIDAD', 'EXPLICACION']]
    for _, row in top_pred.iterrows():
        print(f"\n  ID: {row['OPP_ID']} | Etapa: {row['ETAPA']} | Prob: {row['PROBABILIDAD']:.3f}")
        items = parsear_shap(row['EXPLICACION'])
        for item in items:
            direccion = '▲' if item.get('shap_value', 0) > 0 else '▼'
            print(f"    {direccion} {item.get('feature',''):35} valor={item.get('value')}  SHAP={item.get('shap_value',0):+.3f}")